# Predictive Forecasting of Care Load & Placement Demand
**U.S. Department of Health and Human Services — UAC Program**

In [ ]:
# STEP 0: Setup Environment
!pip install pandas numpy matplotlib seaborn scikit-learn statsmodels openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose

print('All libraries loaded.')

---
### STEP 1: Load & Inspect Dataset

In [ ]:
df = pd.read_excel('hhs_data.xlsx')
df.columns = df.columns.str.strip().str.replace(' ', '_')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')
print('Columns:', list(df.columns))
print('Shape  :', df.shape)
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
print('Missing values (%):')
print((df.isnull().sum() / len(df) * 100).round(2))

---
### STEP 2: Time-Series Preparation

In [ ]:
# Enforce daily frequency and interpolate gaps
df = df.asfreq('D')
df = df.interpolate()
print('After resampling:', df.shape)
print('Date range      :', df.index.min().date(), 'to', df.index.max().date())

---
### STEP 3: Exploratory Data Analysis (EDA)

In [ ]:
target = 'Children_in_HHS_Care'
discharge_target = 'Children_discharged_from_HHS_Care'

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
df[target].plot(ax=axes[0], title='Children in HHS Care (Target)', color='steelblue')
df['Children_transferred_out_of_CBP_custody'].plot(ax=axes[1], title='Daily Transfers into HHS', color='orange')
df[discharge_target].plot(ax=axes[2], title='Daily Discharges from HHS', color='green')
for ax in axes: ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(df.corr().round(2), annot=True, cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.tight_layout(); plt.show()

---
### STEP 4: Time-Series Decomposition

In [ ]:
result = seasonal_decompose(df[target], model='additive', period=7)
result.plot()
plt.suptitle('Trend / Seasonality / Residual Decomposition', y=1.02)
plt.tight_layout(); plt.show()

---
### STEP 5: Feature Engineering

In [ ]:
# Lag Features (t-1, t-7, t-14)
df['lag_1']  = df[target].shift(1)
df['lag_7']  = df[target].shift(7)
df['lag_14'] = df[target].shift(14)

# Rolling Statistics — 7-day and 14-day
df['rolling_mean_7']  = df[target].rolling(7).mean()
df['rolling_std_7']   = df[target].rolling(7).std()
df['rolling_mean_14'] = df[target].rolling(14).mean()
df['rolling_std_14']  = df[target].rolling(14).std()

# Flow-Based Signal (Net Pressure Indicator)
df['net_flow'] = (df['Children_transferred_out_of_CBP_custody'] - df[discharge_target])

# Calendar Effects
df['day_of_week'] = df.index.dayofweek
df['month']       = df.index.month

# Capacity Pressure KPI
df['pressure'] = df['net_flow'].apply(lambda x: 'High' if x > 0 else 'Low')

# Drop NaN from lag/rolling windows
df = df.dropna().copy()
print('After Feature Engineering:', df.shape)

---
### STEP 6: Train–Test Split (Strict Time-Based)

In [ ]:
feature_cols = [c for c in df.columns if c not in [target, discharge_target, 'pressure']]

train_size = int(len(df) * 0.8)
train = df[:train_size]
test  = df[train_size:]

X_train = train[feature_cols];  y_train = train[target]
X_test  = test[feature_cols];   y_test  = test[target]

print('Train period:', train.index.min().date(), '->', train.index.max().date(), '| rows:', len(train))
print('Test  period:', test.index.min().date(),  '->', test.index.max().date(),  '| rows:', len(test))

---
### STEP 7: Evaluation Metric Functions (MAE, RMSE, MAPE)

In [ ]:
def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def evaluate(name, y_true, y_pred):
    mae_  = mean_absolute_error(y_true, y_pred)
    rmse_ = np.sqrt(mean_squared_error(y_true, y_pred))
    mape_ = mape(y_true, y_pred)
    print(f'{name:<24} MAE={mae_:>8.2f}  RMSE={rmse_:>8.2f}  MAPE={mape_:>6.2f}%')
    return mae_, rmse_, mape_

results_summary = []

---
### STEP 8: Baseline Models

In [ ]:
# Naive Persistence (Lag-1)
naive_preds = np.concatenate([[y_test.iloc[0]], y_test.values[:-1]])
mae_n, rmse_n, mape_n = evaluate('Naive (Lag-1)', y_test, naive_preds)
results_summary.append({'Model':'Naive (Lag-1)', 'MAE':mae_n, 'RMSE':rmse_n, 'MAPE':mape_n})

# 7-Day Moving Average
ma_preds = np.array([y_test.values[max(0,i-7):i].mean() if i>0 else y_test.iloc[0] for i in range(len(y_test))])
mae_ma, rmse_ma, mape_ma = evaluate('Moving Average (7d)', y_test, ma_preds)
results_summary.append({'Model':'Moving Average (7d)', 'MAE':mae_ma, 'RMSE':rmse_ma, 'MAPE':mape_ma})

---
### STEP 9: ARIMA & Exponential Smoothing (Statistical Models)

In [ ]:
model_arima     = ARIMA(train[target], order=(5, 1, 0))
model_arima_fit = model_arima.fit()
arima_preds     = model_arima_fit.forecast(steps=len(test)).values
mae_ar, rmse_ar, mape_ar = evaluate('ARIMA(5,1,0)', y_test, arima_preds)
results_summary.append({'Model':'ARIMA(5,1,0)', 'MAE':mae_ar, 'RMSE':rmse_ar, 'MAPE':mape_ar})

In [ ]:
es_model = ExponentialSmoothing(train[target], trend='add', seasonal='add', seasonal_periods=7)
es_fit   = es_model.fit()
es_preds = es_fit.forecast(steps=len(test)).values
mae_es, rmse_es, mape_es = evaluate('Exp Smoothing (HW)', y_test, es_preds)
results_summary.append({'Model':'Exp Smoothing (HW)', 'MAE':mae_es, 'RMSE':rmse_es, 'MAPE':mape_es})

---
### STEP 10: Machine Learning Models — Random Forest & Gradient Boosting

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
mae_rf, rmse_rf, mape_rf = evaluate('Random Forest', y_test, rf_preds)
results_summary.append({'Model':'Random Forest', 'MAE':mae_rf, 'RMSE':rmse_rf, 'MAPE':mape_rf})

In [ ]:
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)
mae_gb, rmse_gb, mape_gb = evaluate('Gradient Boosting', y_test, gb_preds)
results_summary.append({'Model':'Gradient Boosting', 'MAE':mae_gb, 'RMSE':rmse_gb, 'MAPE':mape_gb})

---
### STEP 11: Feature Importance

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.plot(kind='bar', figsize=(12,4), title='Feature Importances — Random Forest', color='steelblue')
plt.ylabel('Importance Score')
plt.tight_layout(); plt.show()

---
### STEP 12: Full Model Comparison Table + Chart

In [ ]:
results_df = pd.DataFrame(results_summary).set_index('Model').round(2)
print('\n=== MODEL COMPARISON TABLE ===')
print(results_df.to_string())
print(f'\nBest by MAE : {results_df["MAE"].idxmin()}')
print(f'Best by MAPE: {results_df["MAPE"].idxmin()}')

In [ ]:
plt.figure(figsize=(13,5))
plt.plot(test.index, y_test.values, label='Actual',         linewidth=2.5, color='black')
plt.plot(test.index, naive_preds,   label='Naive',          linestyle=':', color='gray')
plt.plot(test.index, arima_preds,   label='ARIMA',          linestyle='--', color='orange')
plt.plot(test.index, es_preds,      label='Exp Smoothing',  linestyle='--', color='purple')
plt.plot(test.index, rf_preds,      label='Random Forest',  linestyle='-',  color='steelblue')
plt.plot(test.index, gb_preds,      label='Grad Boosting',  linestyle='-',  color='green')
plt.title('Forecast vs Actual — All Models')
plt.legend(loc='upper left')
plt.tight_layout(); plt.show()

---
### STEP 13: Confidence Interval Visualization (Random Forest — 90% CI)

In [ ]:
tree_preds = np.stack([t.predict(X_test) for t in rf_model.estimators_])
ci_lower   = np.percentile(tree_preds,  5, axis=0)
ci_upper   = np.percentile(tree_preds, 95, axis=0)

plt.figure(figsize=(13,5))
plt.plot(test.index, y_test.values, label='Actual',      linewidth=2, color='black')
plt.plot(test.index, rf_preds,      label='RF Forecast', color='steelblue')
plt.fill_between(test.index, ci_lower, ci_upper, alpha=0.25, color='steelblue', label='90% CI')
plt.title('Random Forest Forecast with 90% Confidence Interval')
plt.legend()
plt.tight_layout(); plt.show()

---
### STEP 14: Horizon Error Analysis (Short vs Medium-Term Reliability)

In [ ]:
horizons = {
    'Short  (1–7 days)' : slice(0, 7),
    'Medium (8–30 days)': slice(7, 30),
    'Long   (31+ days)' : slice(30, None),
}
print('=== HORIZON ERROR ANALYSIS (Random Forest) ===')
for label, sl in horizons.items():
    yt, yp = y_test.values[sl], rf_preds[sl]
    if len(yt) > 0:
        print(f'  {label:<26} MAE={mean_absolute_error(yt,yp):.2f}  MAPE={mape(yt,yp):.2f}%')

---
### STEP 15: Future Forecast — 7 / 14 / 30 Days (Iterative Loop)

In [ ]:
def forecast_future(model, df, feature_cols, target, n_days=7):
    last_vals = list(df[target].values[-14:])
    last_row  = df[feature_cols].iloc[-1].copy()
    preds = []
    for _ in range(n_days):
        row = last_row.copy()
        row['lag_1']           = last_vals[-1]
        row['lag_7']           = last_vals[-7]
        row['lag_14']          = last_vals[-14]
        row['rolling_mean_7']  = np.mean(last_vals[-7:])
        row['rolling_std_7']   = np.std(last_vals[-7:])
        row['rolling_mean_14'] = np.mean(last_vals[-14:])
        row['rolling_std_14']  = np.std(last_vals[-14:])
        pred = model.predict(pd.DataFrame([row]))[0]
        preds.append(round(pred, 0))
        last_vals.append(pred)
    return preds

for horizon in [7, 14, 30]:
    future = forecast_future(rf_model, df, feature_cols, target, n_days=horizon)
    print(f'\n{horizon}-Day Forecast:')
    for i, v in enumerate(future, 1): print(f'  Day {i:>2}: {int(v):,}')

---
### STEP 16: Discharge Demand Forecast

In [ ]:
d_feature_cols   = [c for c in df.columns if c not in [discharge_target, target, 'pressure']]
Xd_train = train[d_feature_cols]; yd_train = train[discharge_target]
Xd_test  = test[d_feature_cols];  yd_test  = test[discharge_target]

discharge_model = RandomForestRegressor(n_estimators=100, random_state=42)
discharge_model.fit(Xd_train, yd_train)
discharge_preds = discharge_model.predict(Xd_test)
print('Discharge Model:')
evaluate('Discharge RF', yd_test, discharge_preds)

plt.figure(figsize=(12,4))
plt.plot(test.index, yd_test.values,   label='Actual Discharges',   color='green')
plt.plot(test.index, discharge_preds,  label='Forecast Discharges',  color='orange', linestyle='--')
plt.title('Discharge Demand Forecast')
plt.legend(); plt.tight_layout(); plt.show()

---
### STEP 17: KPI Summary Dashboard

In [ ]:
accuracy     = 100 - (mae_rf / y_test.mean() * 100)
breach_thr   = float(df[target].quantile(0.90))
breach_prob  = (rf_preds > breach_thr).mean() * 100
high_pressure= (df['pressure'] == 'High').sum()
stability    = max(0.0, 1 - (rmse_rf / y_test.std()))
future30     = forecast_future(rf_model, df, feature_cols, target, n_days=30)
surge_day    = next((i+1 for i,v in enumerate(future30) if v > breach_thr), None)

print('=' * 55)
print('              PROJECT KPI SUMMARY')
print('=' * 55)
print(f'  Forecast Accuracy (RF)       : {accuracy:.2f}%')
print(f'  RF MAE                       : {mae_rf:.2f}')
print(f'  RF RMSE                      : {rmse_rf:.2f}')
print(f'  RF MAPE                      : {mape_rf:.2f}%')
print(f'  Capacity Breach Threshold    : {breach_thr:,.0f} children')
print(f'  Capacity Breach Probability  : {breach_prob:.1f}%')
print(f'  Forecast Stability Index     : {stability:.3f}')
print(f'  High Pressure Days (hist.)   : {high_pressure} / {len(df)}')
print(f'  Surge Lead Time              : {str(surge_day)+" days" if surge_day else "No surge in 30d"}')
print(f'  Avg Net Flow                 : {df["net_flow"].mean():.2f}')
print('=' * 55)

---
### STEP 18: Launch Streamlit Dashboard

Run the interactive dashboard from your terminal:
```bash
streamlit run app.py
```